# 04 · Rotulagem fraca com LLM

Rotula ~12 mil comentários com um LLM aberto em 4 bits (GPU T4). Primeiro mede a concordância com o gabarito de **validação**; só depois rotula o conjunto grande.

Ambiente: **GPU T4** (Ambiente de execução → Alterar tipo).

In [ ]:
# Configuração: no Colab, clona o repositório e instala o pacote `absa`
import sys
if "google.colab" in sys.modules:
    !git clone -q https://github.com/vnevesl/pln-absa-olist.git 2>/dev/null || git -C pln-absa-olist pull -q
    %cd pln-absa-olist
    !pip -q install -e ".[llm]"
    # Guarda cache do LLM e modelos no Drive para sobreviver à queda da sessão
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/pln-absa-olist/llm_cache /content/drive/MyDrive/pln-absa-olist/models
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/llm_cache data/llm_cache
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/models models

In [ ]:
import pandas as pd
from absa.config import ANNOTATED, ASPECTS, INTERIM, LABEL_COLUMNS, SEED
from absa.data import load_reviews
from absa.llm_labeler import hf_generator, label_texts
from absa.metrics import per_aspect_report, tuple_f1
from absa.preprocess import clean_light
from absa.sampling import load_gold

dev, test = load_gold()
gold_ids = set(dev.review_id) | set(test.review_id)

# Few-shot: exemplos fixos da validação cobrindo aspectos variados
def _ex(r):
    return {"texto": clean_light(r.texto), **{a: r[f"asp_{a}"] for a in ASPECTS}}
exemplos = [_ex(r) for _, r in dev.sample(6, random_state=SEED).iterrows()]
exemplos_ids = set(dev.sample(6, random_state=SEED).review_id)

In [ ]:
generate = hf_generator("Qwen/Qwen2.5-7B-Instruct")  # alternativa: openai_compatible_generator()

## 1) Concordância LLM × humano na validação

In [ ]:
aval = dev[~dev.review_id.isin(exemplos_ids)]
rot, brutos = label_texts(aval.texto.map(clean_light).tolist(), generate, exemplos, cache_name="qwen7b_dev")
validos = [r is not None for r in rot]
print(f"respostas válidas: {sum(validos)}/{len(rot)}")
pred = pd.DataFrame([r for r in rot if r]).rename(columns=lambda a: f"asp_{a}")
ref = aval[validos].reset_index(drop=True)
print("F1 tupla:", round(tuple_f1(ref, pred), 3))
per_aspect_report(ref, pred).round(3)

Se a concordância estiver baixa, ajuste os exemplos ou o texto do prompt em `src/absa/llm_labeler.py` e repita a etapa 1 **antes** de rotular o conjunto grande.

## 2) Rotulagem do conjunto de treino (sem os comentários do gabarito)

In [ ]:
df = load_reviews()
pool = df[~df.review_id.isin(gold_ids) & (df.n_palavras >= 3)].drop_duplicates("texto")
# Estratifica por nota para não ficar dominado por 5 estrelas
treino = pd.concat(g.sample(min(len(g), 2400), random_state=SEED) for _, g in pool.groupby("nota"))
rot, _ = label_texts(treino.texto.map(clean_light).tolist(), generate, exemplos, cache_name="qwen7b_treino")
treino = treino.assign(**{f"asp_{a}": [r[a] if r else None for r in rot] for a in ASPECTS})
treino = treino.dropna(subset=LABEL_COLUMNS)
treino[["review_id", "texto", "nota", "atrasado"] + LABEL_COLUMNS].to_csv(INTERIM / "rotulos_llm.csv", index=False)
print(len(treino))
treino[LABEL_COLUMNS].apply(lambda s: s.value_counts()).fillna(0).astype(int)